# Entraînement multi-horizon (15 / 30 / 60 min) + météo — PFE

Le notebook `03_model_comparison.ipynb` a montré qu'à horizon = 1 collecte
(~5 min), aucun modèle testé ne bat significativement la baseline de
persistance. C'est attendu : sur un horizon aussi court, la meilleure
prédiction est presque toujours "la valeur actuelle".

L'intérêt d'un modèle apparaît surtout à **horizon plus long** (15, 30,
60 min), où la persistance simple devient moins fiable (plus le temps
passe, plus la station a une chance d'avoir changé). Ce notebook :

1. Réutilise les features du notebook 03 (lags, heure, jour, météo)
2. Entraîne **un modèle XGBoost par horizon** (15 / 30 / 60 min)
3. Compare chaque modèle à sa propre baseline de persistance
4. Sauvegarde uniquement les modèles qui battent réellement leur baseline
   (`ml_models/xgb_h{horizon}.joblib`), et journalise le choix dans
   `ml_models/metrics.json` pour que l'API sache, horizon par horizon,
   s'il faut utiliser le modèle ou se rabattre sur la persistance.

C'est une démarche plus rigoureuse qu'un seul modèle "par défaut" : chaque
horizon garde la meilleure des deux stratégies, avec les chiffres à l'appui.


In [1]:
import os
import json
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()
DB_URL = (
    f"postgresql+psycopg2://{os.environ['DB_USER']}:{os.environ['DB_PASSWORD']}"
    f"@{os.environ['DB_HOST']}:{os.environ['DB_PORT']}/{os.environ['DB_NAME']}"
)
engine = create_engine(DB_URL)

status = pd.read_sql("""
    select station_id, collected_at, num_bikes_available, num_docks_available
    from dbt_dev.silver_station_status
    order by station_id, collected_at
""", engine, parse_dates=["collected_at"])

weather = pd.read_sql("""
    select collected_at, temperature_c, precipitation_mm, wind_speed_kmh
    from dbt_dev.silver_weather
    order by collected_at
""", engine, parse_dates=["collected_at"])

print(f"Status : {len(status):,} lignes, {status['station_id'].nunique()} stations")
print(f"Météo  : {len(weather):,} relevés")


Status : 113,631 lignes, 1516 stations
Météo  : 29 relevés


## 1. Intervalle de collecte réel

Le nombre de "pas" correspondant à 15/30/60 minutes dépend de la fréquence
réelle de collecte (en théorie 5 min, en pratique parfois irrégulier à
cause de la collecte qui tourne sur la machine de dev). On calcule
l'intervalle médian observé plutôt que de supposer 5 min en dur.

In [2]:
diffs = (
    status.sort_values(["station_id", "collected_at"])
    .groupby("station_id")["collected_at"]
    .diff()
    .dt.total_seconds() / 60
)
STEP_MINUTES = float(diffs.median())
print(f"Intervalle de collecte médian observé : {STEP_MINUTES:.1f} min")

HORIZONS_MIN = [15, 30, 60]
STEPS_BY_HORIZON = {
    h: max(1, round(h / STEP_MINUTES)) for h in HORIZONS_MIN
}
print("Pas (nombre de collectes) par horizon :", STEPS_BY_HORIZON)


Intervalle de collecte médian observé : 5.0 min
Pas (nombre de collectes) par horizon : {15: 3, 30: 6, 60: 12}


## 2. Fusion météo + features communes

Mêmes features que le notebook 03 (lags, heure, jour, météo), la seule
chose qui change d'un horizon à l'autre est la cible (`target`), décalée
de `steps` collectes dans le futur au lieu d'une seule.

In [3]:
status_sorted = status.sort_values("collected_at")
weather_sorted = weather.sort_values("collected_at")

df = pd.merge_asof(
    status_sorted, weather_sorted,
    on="collected_at",
    direction="backward",
    tolerance=pd.Timedelta("2h"),
)

N_LAGS = 3
df = df.sort_values(["station_id", "collected_at"]).reset_index(drop=True)
df["hour"] = df["collected_at"].dt.hour
df["day_of_week"] = df["collected_at"].dt.dayofweek

for lag in range(1, N_LAGS + 1):
    df[f"lag_{lag}"] = df.groupby("station_id")["num_bikes_available"].shift(lag)

for col in ["temperature_c", "precipitation_mm", "wind_speed_kmh"]:
    df[col] = df[col].ffill().bfill()

FEATURES = (
    ["hour", "day_of_week"]
    + [f"lag_{lag}" for lag in range(1, N_LAGS + 1)]
    + ["temperature_c", "precipitation_mm", "wind_speed_kmh"]
)
print("Features :", FEATURES)


Features : ['hour', 'day_of_week', 'lag_1', 'lag_2', 'lag_3', 'temperature_c', 'precipitation_mm', 'wind_speed_kmh']


## 3. Entraînement d'un modèle par horizon

Pour chaque horizon (15/30/60 min) :
- `target` = nombre de vélos disponibles `steps` collectes plus tard
- Validation croisée temporelle (`TimeSeriesSplit`, 3 découpages — on en
  utilise moins qu'au notebook 03 pour limiter le temps d'entraînement
  avec 3 horizons à comparer)
- Comparaison XGBoost vs baseline de persistance (`lag_1`, c'est-à-dire
  "la station n'a pas changé depuis maintenant")

In [4]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor

N_SPLITS = 3
results_by_horizon = {}

for horizon_min, steps in STEPS_BY_HORIZON.items():
    d = df.copy()
    d["target"] = d.groupby("station_id")["num_bikes_available"].shift(-steps)
    dataset = d.dropna(subset=[f"lag_{l}" for l in range(1, N_LAGS + 1)] + ["target"])
    dataset = dataset.sort_values("collected_at").reset_index(drop=True)

    X = dataset[FEATURES]
    y = dataset["target"]
    baseline_pred = dataset["lag_1"]

    tscv = TimeSeriesSplit(n_splits=N_SPLITS)
    model_maes, baseline_maes = [], []

    for train_idx, test_idx in tscv.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        model = XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42)
        model.fit(X_train, y_train)
        pred = model.predict(X_test)

        model_maes.append(mean_absolute_error(y_test, pred))
        baseline_maes.append(mean_absolute_error(y_test, baseline_pred.iloc[test_idx]))

    mae_model = float(np.mean(model_maes))
    mae_baseline = float(np.mean(baseline_maes))
    use_model = mae_model < mae_baseline

    results_by_horizon[horizon_min] = {
        "steps": steps,
        "mae_model": round(mae_model, 3),
        "mae_baseline": round(mae_baseline, 3),
        "use_model": use_model,
        "n_train_rows": int(len(dataset)),
    }
    verdict = "modèle retenu" if use_model else "baseline retenue (modèle ne bat pas la persistance)"
    print(f"Horizon {horizon_min:>2} min ({steps} pas) : "
          f"MAE modèle={mae_model:.3f} vs MAE baseline={mae_baseline:.3f} -> {verdict}")


Horizon 15 min (3 pas) : MAE modèle=3.775 vs MAE baseline=2.029 -> baseline retenue (modèle ne bat pas la persistance)
Horizon 30 min (6 pas) : MAE modèle=4.623 vs MAE baseline=3.082 -> baseline retenue (modèle ne bat pas la persistance)
Horizon 60 min (12 pas) : MAE modèle=5.511 vs MAE baseline=4.936 -> baseline retenue (modèle ne bat pas la persistance)


## 4. Résumé et sauvegarde

Pour chaque horizon où le modèle bat réellement la baseline, on réentraîne
sur toutes les données disponibles et on sauvegarde le modèle. Sinon, on
ne sauvegarde rien : l'API se rabattra sur la persistance pour cet
horizon (voir `ml_models/metrics.json`), ce qui est un choix honnête et
documenté plutôt qu'un modèle sur-vendu.

In [5]:
summary = pd.DataFrame(results_by_horizon).T
summary.index.name = "horizon_min"
print(summary)

os.makedirs("../ml_models", exist_ok=True)

for horizon_min, steps in STEPS_BY_HORIZON.items():
    info = results_by_horizon[horizon_min]
    if not info["use_model"]:
        continue
    d = df.copy()
    d["target"] = d.groupby("station_id")["num_bikes_available"].shift(-steps)
    dataset = d.dropna(subset=[f"lag_{l}" for l in range(1, N_LAGS + 1)] + ["target"])
    final_model = XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42)
    final_model.fit(dataset[FEATURES], dataset["target"])
    path = f"../ml_models/xgb_h{horizon_min}.joblib"
    import joblib
    joblib.dump(final_model, path)
    print(f"Horizon {horizon_min} min -> sauvegardé dans {path}")

metrics_path = "../ml_models/metrics.json"
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump({
        "step_minutes_observed": STEP_MINUTES,
        "features": FEATURES,
        "horizons": results_by_horizon,
    }, f, ensure_ascii=False, indent=2)
print(f"Métriques sauvegardées dans {metrics_path}")


            steps mae_model mae_baseline use_model n_train_rows
horizon_min                                                    
15              3     3.775        2.029     False       104535
30              6     4.623        3.082     False        99990
60             12     5.511        4.936     False        90900
Métriques sauvegardées dans ../ml_models/metrics.json


## Pour le mémoire

- Un modèle unique "par défaut" cache souvent le fait qu'il n'apporte rien
  à court terme : ici, le choix modèle vs baseline est **fait horizon par
  horizon, avec les chiffres à l'appui**, pas par défaut.
- La météo est intégrée comme feature (température, précipitations, vent),
  testée sur chaque horizon plutôt que supposée utile a priori.
- L'API (`api/main.py`) lit `ml_models/metrics.json` au démarrage et choisit
  automatiquement, pour chaque horizon demandé, le modèle appris ou la
  persistance — un exemple concret de "graceful degradation" plutôt que
  d'imposer un modèle qui ne serait pas meilleur que le bon sens.
